<a href="https://colab.research.google.com/github/ingkapat/Thai-Scam-Call-Detector/blob/main/prepare_split.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Cell 2: Paths config
import os

# ปรับให้ตรงกับที่อัป zip ไว้ใน Drive ของคุณ
DRIVE_BASE  = '/content/drive/MyDrive/ThaiScamCall'
ZIP_PATH    = f'{DRIVE_BASE}/mp3_15s.zip'
EXTRACT_DIR = '/content/data_15s'           # local Colab (เร็ว)
SPLITS_DIR  = f'{DRIVE_BASE}/splits'        # บันทึก split csv ที่นี่

os.makedirs(SPLITS_DIR, exist_ok=True)
assert os.path.exists(ZIP_PATH), f'ไม่เจอ {ZIP_PATH} — อัป mp3_15s.zip ขึ้น Drive ก่อน'
print('OK')

OK


In [3]:
# Cell 3: Extract zip ไปยัง local Colab
import zipfile

if not os.path.exists(EXTRACT_DIR) or len(os.listdir(EXTRACT_DIR)) < 1000:
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)

n = len(os.listdir(EXTRACT_DIR))
print(f'Extracted files: {n}')
assert n > 20000, 'น่าจะแตกไม่ครบ ลองรันใหม่'

Extracted files: 21287


In [4]:
# Cell 4: Build manifest จากชื่อไฟล์ (label ฝังในชื่อ _label0 / _label1)
import glob, re
import pandas as pd

files = sorted(glob.glob(f'{EXTRACT_DIR}/**/*.mp3', recursive=True))

def parse_label(p):
    m = re.search(r'_label([01])\.mp3$', p)
    return int(m.group(1)) if m else None

df = pd.DataFrame({
    'filename': [os.path.basename(p) for p in files],
    'label':    [parse_label(p) for p in files],
})
df = df.dropna(subset=['label']).copy()
df['label'] = df['label'].astype(int)

print('Total:', len(df))
print('Class distribution:')
print(df.label.value_counts().rename({0:'not_scam', 1:'scam'}))

Total: 21287
Class distribution:
label
not_scam    11156
scam        10131
Name: count, dtype: int64


In [5]:
# Cell 5: Stratified split 80/10/10 และ save
from sklearn.model_selection import train_test_split

# 80 / 10 / 10  stratified
train_df, tmp = train_test_split(
    df, test_size=0.20, stratify=df.label, random_state=42
)
val_df, test_df = train_test_split(
    tmp, test_size=0.50, stratify=tmp.label, random_state=42
)

for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    out = f'{SPLITS_DIR}/{name}.csv'
    d.to_csv(out, index=False)
    print(f'{name:5s} n={len(d):>6} | {d.label.value_counts().to_dict()}  ->  {out}')

train n= 17029 | {0: 8924, 1: 8105}  ->  /content/drive/MyDrive/ThaiScamCall/splits/train.csv
val   n=  2129 | {0: 1116, 1: 1013}  ->  /content/drive/MyDrive/ThaiScamCall/splits/val.csv
test  n=  2129 | {0: 1116, 1: 1013}  ->  /content/drive/MyDrive/ThaiScamCall/splits/test.csv
